# Notebook 067: Distributed Surrogate Key Generation: monotonically_increasing_id vs xxhash64 vs UUIDs

**Module:** Module 07: Lakehouse Data Modeling, CDC & SCD Implementation  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 23: Interview Questions & Video 28: Data Modeling E2E Course  

---

## 1. Business Scenario & Real-World Context
A lakehouse dimensional model builds customer and account dimension tables. Business analysts require synthetic integer or hash surrogate keys to decouple dimensional records from operational natural keys and enable SCD Type 2 version tracking.

## 2. Technical Requirement & Problem Statement
Evaluate distributed surrogate key generation strategies in PySpark: (1) `monotonically_increasing_id()`, (2) Deterministic cryptographic hashing (`xxhash64` / `sha2`), and (3) Universally unique identifiers (`uuid`). Contrast deterministic replayability, performance, and storage footprint.

## 3. Spark & Databricks Architecture Notes
`monotonically_increasing_id()` generates 64-bit integers where the upper 33 bits represent the partition ID and the lower 31 bits represent the row ID within the partition. While fast, IDs are NOT contiguous and change if the partition count changes. `xxhash64(concat_ws())` produces deterministic 64-bit hash keys that remain identical across pipeline replays.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_067_Distributed_Surrogate_Key_Generation_monotonically_increasing_id_vs_xxhash64_vs_UUIDs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Customer Natural Keys Across Partitions
Create customers distributed across multiple partitions.

In [ ]:
natural_customers = [
    ("NAT-1001", "Alice Smith", "2023-01-10"),
    ("NAT-1002", "Bob Jones", "2023-02-15"),
    ("NAT-1003", "Charlie Day", "2023-03-20"),
    ("NAT-1004", "Diana Prince", "2023-04-25")
]

schema = "natural_key STRING, name STRING, created_at STRING"
df_cust = spark.createDataFrame(natural_customers, schema).repartition(2)
print(f"Partitions: {df_cust.rdd.getNumPartitions()}")

## 6. Comparing Key Generation Strategies
Implement monotonically_increasing_id, xxhash64, and UUID generation.

In [ ]:
df_keys = (
    df_cust
    # 1. Monotonically increasing ID (Fast, non-contiguous, partition-dependent)
    .withColumn("surrogate_monotonic_id", F.monotonically_increasing_id())
    # 2. Deterministic 64-bit integer hash (Replayable, 100% deterministic, 8 bytes)
    .withColumn("surrogate_xxhash64", F.xxhash64(F.col("natural_key"), F.col("created_at")))
    # 3. MD5/SHA256 string hash (Deterministic, 32/64 character string, higher storage cost)
    .withColumn("surrogate_sha256", F.sha2(F.col("natural_key"), 256))
    # 4. Random UUID (Unique, non-deterministic across pipeline reruns)
    .withColumn("surrogate_uuid", F.expr("uuid()"))
)

df_keys.select("natural_key", "surrogate_monotonic_id", "surrogate_xxhash64", "surrogate_uuid").show(truncate=False)

## 7. Deterministic Replayability Verification
Verify that xxhash64 produces the exact same key upon re-running the pipeline.

In [ ]:
# Re-evaluating xxhash64 on the same inputs yields identical hash values
df_recheck = df_cust.withColumn("recheck_hash", F.xxhash64(F.col("natural_key"), F.col("created_at")))
diff_count = df_keys.join(df_recheck, on="natural_key").filter(F.col("surrogate_xxhash64") != F.col("recheck_hash")).count()

assert diff_count == 0, "Deterministic hash mismatch detected!"
print("xxhash64 verified as 100% deterministic across distributed runs!")

## 9. Key Architectural Takeaways
- **`xxhash64(concat_ws('||', *keys))` is the recommended surrogate key strategy for distributed Lakehouses: 64-bit integer, fast, and 100% deterministic.**
- **`monotonically_increasing_id()` generates gaps between partitions and changes if data is repartitioned.**
- **UUIDs are non-deterministic and prevent idempotent pipeline reruns.**

## 10. Senior Data Engineering Interview Questions
1. How does `monotonically_increasing_id()` construct its 64-bit integer across partitions?
2. Why is `xxhash64` preferred over UUIDs for dimensional surrogate keys in Lakehouse architectures?
3. What is the risk of hash collisions in 64-bit hashes on tables with billions of rows?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
